**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [2]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

Progetto: floods


TRAINING

In [4]:
encoders_train_func = project.new_function(
    name="encoders_train-job-v58",
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="pretrain_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")



2026-08-12 14:04:38,904 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:04:43,914 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:04:49,061 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:04:54,076 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:04:59,091 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:05:04,106 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14:05:09,121 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run f68dd9081f9446648c1decec94a3331f to finish...
2026-08-12 14

BUILD: COMPLETED


In [ ]:
# setup ambiente
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

parametri = {
    "epochs": 1, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # ottiche               
    "mamba": False, 
    "workers": 0
}

print(f"PARAMETRI: {parametri}")

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",
    # local_execution= True,                                # 1x = 1 gpu
    wait=True
)

print(f"STATO FINALE: {run_train_encoders.status.state}")
print(run_train_encoders.logs())

2026-08-12 14:05:31,440 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...


PARAMETRI: {'epochs': 1, 'batch_size': 16, 'lr': 0.0001, 'weight_decay': 0.0001, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'mamba': False, 'workers': 0}


2026-08-12 14:05:36,559 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:05:41,573 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:05:46,586 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:05:51,601 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:05:56,615 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:06:03,733 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14:06:08,751 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 9e641001f3824566adc764122c5be258 to finish...
2026-08-12 14

RISULTATI

In [ ]:
# salvataggio log
print("SALVATAGGIO METRICHE")

path_s1 = project.get_artifact("metrics-s1").download(overwrite=True)
path_s2 = project.get_artifact("metrics-s2").download(overwrite=True)

# risultati grafici
df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title('Pre-training SAR')
ax1.set_xlabel('Epoche')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title('Pre-training OTTICO')
ax2.set_xlabel('Epoche')
ax2.grid(True)

plt.show()
